## Practice notebook for polars and statistics

In [1]:
import polars as pl
import numpy as np
from typing import List

In [2]:
# 1. logical plan: pl.scan_csv() creates a LazyFrame.This is a non-materialized query plan 
# that enables optimization.
# 2. evaluation: .collect() optimizes the Logical Plan into a physical Plan and executes it.
# The result is a DataFrame

lf = pl.scan_csv("abalone_data.csv")
print(type(lf))
df = lf.collect()
print(type(df))

<class 'polars.lazyframe.frame.LazyFrame'>
<class 'polars.dataframe.frame.DataFrame'>


In [3]:
print(lf.head().collect())

shape: (5, 9)
┌────────┬────────┬──────────┬────────┬───┬────────────────┬────────────────┬──────────────┬───────┐
│ gender ┆ Length ┆ Diameter ┆ Height ┆ … ┆ Shucked weight ┆ Viscera weight ┆ Shell weight ┆ Rings │
│ ---    ┆ ---    ┆ ---      ┆ ---    ┆   ┆ ---            ┆ ---            ┆ ---          ┆ ---   │
│ str    ┆ f64    ┆ f64      ┆ f64    ┆   ┆ f64            ┆ f64            ┆ f64          ┆ i64   │
╞════════╪════════╪══════════╪════════╪═══╪════════════════╪════════════════╪══════════════╪═══════╡
│ M      ┆ 0.455  ┆ 0.365    ┆ 0.095  ┆ … ┆ 0.2245         ┆ 0.101          ┆ 0.15         ┆ 15    │
│ M      ┆ 0.35   ┆ 0.265    ┆ 0.09   ┆ … ┆ 0.0995         ┆ 0.0485         ┆ 0.07         ┆ 7     │
│ F      ┆ 0.53   ┆ 0.42     ┆ 0.135  ┆ … ┆ 0.2565         ┆ 0.1415         ┆ 0.21         ┆ 9     │
│ M      ┆ 0.44   ┆ 0.365    ┆ 0.125  ┆ … ┆ 0.2155         ┆ 0.114          ┆ 0.155        ┆ 10    │
│ I      ┆ 0.33   ┆ 0.255    ┆ 0.08   ┆ … ┆ 0.0895         ┆ 0.0395         ┆

In [4]:
print(f"Available columns: {lf.collect_schema().names()}")

Available columns: ['gender', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']


In [5]:
lf.describe()

statistic,gender,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""4177""",4177.0,4177.0,4177.0,4177.0,4177.0,4177.0,4177.0,4177.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,0.523992,0.407881,0.139516,0.828742,0.359367,0.180594,0.238831,9.933684
"""std""",null,0.120093,0.09924,0.041827,0.490389,0.221963,0.109614,0.139203,3.224169
"""min""","""F""",0.075,0.055,0.0,0.002,0.001,0.0005,0.0015,1.0
"""25%""",null,0.45,0.35,0.115,0.4415,0.186,0.0935,0.13,8.0
"""50%""",null,0.545,0.425,0.14,0.7995,0.336,0.171,0.234,9.0
"""75%""",null,0.615,0.48,0.165,1.153,0.502,0.253,0.329,11.0
"""max""","""M""",0.815,0.65,1.13,2.8255,1.488,0.76,1.005,29.0


In [6]:
print(
    lf.group_by("gender")
      .agg(pl.len(), pl.mean("Height"), pl.mean("Whole weight"))
      .collect()
)

shape: (3, 4)
┌────────┬──────┬──────────┬──────────────┐
│ gender ┆ len  ┆ Height   ┆ Whole weight │
│ ---    ┆ ---  ┆ ---      ┆ ---          │
│ str    ┆ u32  ┆ f64      ┆ f64          │
╞════════╪══════╪══════════╪══════════════╡
│ M      ┆ 1528 ┆ 0.151381 ┆ 0.991459     │
│ I      ┆ 1342 ┆ 0.107996 ┆ 0.431363     │
│ F      ┆ 1307 ┆ 0.158011 ┆ 1.046532     │
└────────┴──────┴──────────┴──────────────┘


In [7]:
print(
    lf.filter(pl.col("gender") == "M", pl.col("Shell weight") > 0.15)
    .sort(pl.col("Length"), descending=True)
    .head(20)
    .collect()
)

shape: (20, 9)
┌────────┬────────┬──────────┬────────┬───┬────────────────┬────────────────┬──────────────┬───────┐
│ gender ┆ Length ┆ Diameter ┆ Height ┆ … ┆ Shucked weight ┆ Viscera weight ┆ Shell weight ┆ Rings │
│ ---    ┆ ---    ┆ ---      ┆ ---    ┆   ┆ ---            ┆ ---            ┆ ---          ┆ ---   │
│ str    ┆ f64    ┆ f64      ┆ f64    ┆   ┆ f64            ┆ f64            ┆ f64          ┆ i64   │
╞════════╪════════╪══════════╪════════╪═══╪════════════════╪════════════════╪══════════════╪═══════╡
│ M      ┆ 0.78   ┆ 0.6      ┆ 0.21   ┆ … ┆ 1.1945         ┆ 0.5745         ┆ 0.6745       ┆ 11    │
│ M      ┆ 0.775  ┆ 0.63     ┆ 0.25   ┆ … ┆ 1.3485         ┆ 0.76           ┆ 0.578        ┆ 12    │
│ M      ┆ 0.775  ┆ 0.57     ┆ 0.22   ┆ … ┆ 0.735          ┆ 0.4755         ┆ 0.6585       ┆ 17    │
│ M      ┆ 0.77   ┆ 0.62     ┆ 0.195  ┆ … ┆ 1.1155         ┆ 0.6415         ┆ 0.642        ┆ 12    │
│ M      ┆ 0.77   ┆ 0.605    ┆ 0.175  ┆ … ┆ 0.8005         ┆ 0.526          

In [8]:
avg_heavy_m_height = (
    lf.filter(pl.col("gender") == "M")
    .sort(pl.col("Shucked weight"), descending=True)
    .head(20)
    .select(pl.mean(("Height")))
    .collect()
)
avg_light_m_height = (
    lf.filter(pl.col("gender") == "M")
    .sort(pl.col("Shucked weight"), descending=False)
    .head(20)
    .select(pl.mean("Height"))
    .collect()
)

print(f"The average height of the 20 heaviest is {round(avg_heavy_m_height.item(), 2)}")
print(f"The average height of the 20 lightest is {round(avg_light_m_height.item(), 2)}")

The average height of the 20 heaviest is 0.22
The average height of the 20 lightest is 0.06


In [9]:
# Covariance calculations
cov_mm_length_height = (
    lf.select(pl.cov(pl.col("Length"), pl.col("Height")))
)
cov_cm_length_height = (
    lf.select(pl.cov(pl.col("Length")*10, pl.col("Height") * 10))
)

print(f"Cov with mm as unit: {round(cov_mm_length_height.collect().item(),5)}")
print(f"Cov with cm as unit: {round(cov_cm_length_height.collect().item(),5)}")

Cov with mm as unit: 0.00416
Cov with cm as unit: 0.41569


In [10]:
# Correlation calculations
corr_mm_length_height = (
    lf.select(pl.corr(pl.col("Length"), pl.col("Height")))
)
corr_cm_length_height = (
    lf.select(pl.corr(pl.col("Length") * 10, pl.col("Height") * 10))
)

print(f"Corr with mm as unit: {round(corr_mm_length_height.collect().item(),5)}")
print(f"Corr with cm as unit: {round(corr_cm_length_height.collect().item(),5)}")
# It is standardized

Corr with mm as unit: 0.82755
Corr with cm as unit: 0.82755


In [11]:
print(lf.collect_schema().names())

['gender', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']


In [12]:
cols = ["Infant", "Length", "Height", "Diameter", "Whole weight", "Rings"]
lf_infant_column = (
    lf.select(pl.col(["gender", "Length", "Height", "Diameter", "Whole weight", "Rings"]), 
        pl.when(pl.col("gender") == "I").then(pl.lit(1)).otherwise(pl.lit(0)).alias("Infant"))
        .drop(pl.col("gender"))
        .select(pl.col(cols))
)
lf_infant_column_density = (
    lf_infant_column
        .filter((pl.col("Length") != 0) & (pl.col("Height") != 0) & (pl.col("Diameter") != 0))
        .with_columns((pl.col("Whole weight") / 
            (pl.col("Length") * pl.col("Height") * pl.col("Diameter")))
        .alias("Density")
    )
)

print(lf_infant_column_density.collect())

shape: (4_175, 7)
┌────────┬────────┬────────┬──────────┬──────────────┬───────┬───────────┐
│ Infant ┆ Length ┆ Height ┆ Diameter ┆ Whole weight ┆ Rings ┆ Density   │
│ ---    ┆ ---    ┆ ---    ┆ ---      ┆ ---          ┆ ---   ┆ ---       │
│ i32    ┆ f64    ┆ f64    ┆ f64      ┆ f64          ┆ i64   ┆ f64       │
╞════════╪════════╪════════╪══════════╪══════════════╪═══════╪═══════════╡
│ 0      ┆ 0.455  ┆ 0.095  ┆ 0.365    ┆ 0.514        ┆ 15    ┆ 32.578813 │
│ 0      ┆ 0.35   ┆ 0.09   ┆ 0.265    ┆ 0.2255       ┆ 7     ┆ 27.014076 │
│ 0      ┆ 0.53   ┆ 0.135  ┆ 0.42     ┆ 0.677        ┆ 9     ┆ 22.528368 │
│ 0      ┆ 0.44   ┆ 0.125  ┆ 0.365    ┆ 0.516        ┆ 10    ┆ 25.703611 │
│ 1      ┆ 0.33   ┆ 0.08   ┆ 0.255    ┆ 0.205        ┆ 7     ┆ 30.451575 │
│ …      ┆ …      ┆ …      ┆ …        ┆ …            ┆ …     ┆ …         │
│ 0      ┆ 0.565  ┆ 0.165  ┆ 0.45     ┆ 0.887        ┆ 11    ┆ 21.143589 │
│ 0      ┆ 0.59   ┆ 0.135  ┆ 0.44     ┆ 0.966        ┆ 10    ┆ 27.563773 │
│ 0    

In [13]:
rings_mean = lf.select(pl.mean("Rings")).collect().item()
rings_std = lf.select(pl.std("Rings")).collect().item()
print(f"average ring count: {rings_mean}, std: {rings_std}")

average ring count: 9.933684462532918, std: 3.224169032068128


In [14]:
lf_infant_column_density_filtered = (
    lf_infant_column_density
    .filter(
        (pl.col("Rings") > rings_mean - 2 * rings_std) & 
        (pl.col("Rings") < rings_mean + 2 * rings_std)
        ).drop("Rings")
)

print(lf_infant_column_density_filtered.collect())

shape: (3_964, 6)
┌────────┬────────┬────────┬──────────┬──────────────┬───────────┐
│ Infant ┆ Length ┆ Height ┆ Diameter ┆ Whole weight ┆ Density   │
│ ---    ┆ ---    ┆ ---    ┆ ---      ┆ ---          ┆ ---       │
│ i32    ┆ f64    ┆ f64    ┆ f64      ┆ f64          ┆ f64       │
╞════════╪════════╪════════╪══════════╪══════════════╪═══════════╡
│ 0      ┆ 0.455  ┆ 0.095  ┆ 0.365    ┆ 0.514        ┆ 32.578813 │
│ 0      ┆ 0.35   ┆ 0.09   ┆ 0.265    ┆ 0.2255       ┆ 27.014076 │
│ 0      ┆ 0.53   ┆ 0.135  ┆ 0.42     ┆ 0.677        ┆ 22.528368 │
│ 0      ┆ 0.44   ┆ 0.125  ┆ 0.365    ┆ 0.516        ┆ 25.703611 │
│ 1      ┆ 0.33   ┆ 0.08   ┆ 0.255    ┆ 0.205        ┆ 30.451575 │
│ …      ┆ …      ┆ …      ┆ …        ┆ …            ┆ …         │
│ 0      ┆ 0.565  ┆ 0.165  ┆ 0.45     ┆ 0.887        ┆ 21.143589 │
│ 0      ┆ 0.59   ┆ 0.135  ┆ 0.44     ┆ 0.966        ┆ 27.563773 │
│ 0      ┆ 0.6    ┆ 0.205  ┆ 0.475    ┆ 1.176        ┆ 20.12837  │
│ 0      ┆ 0.625  ┆ 0.15   ┆ 0.485    ┆ 1.09

In [15]:
print(
    lf_infant_column_density_filtered.select(pl.col("Density").max()).collect().item()
)
mean_density = lf_infant_column_density_filtered.select(pl.col("Density").mean()).collect().item()
std_density = lf_infant_column_density_filtered.select(pl.col("Density").std()).collect().item()
print(mean_density, std_density)


245.28752087807206
24.594672232620717 5.265682533897975


In [16]:
def normalize_col(lf: pl.LazyFrame, columns: List[str]) -> pl.LazyFrame:
    return lf.with_columns([
        (
            (pl.col(c) - pl.col(c).min()) 
            / (pl.col(c).max() - pl.col(c).min())
        ).alias(f"{c}_normalized")
        for c in columns
    ])

In [17]:
columns_to_normalize = ["Length", "Height", "Diameter", "Whole weight", "Density"]
# columns_to_normalize = ["Density"]
lf_normalized = (
    normalize_col(lf_infant_column_density_filtered, columns_to_normalize)
    .drop(columns_to_normalize)
)
print(lf_normalized.head().collect())

shape: (5, 6)
┌────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┐
│ Infant ┆ Length_normaliz ┆ Height_normaliz ┆ Diameter_normal ┆ Whole weight_no ┆ Density_normali │
│ ---    ┆ ed              ┆ ed              ┆ ized            ┆ rmalized        ┆ zed             │
│ i32    ┆ ---             ┆ ---             ┆ ---             ┆ ---             ┆ ---             │
│        ┆ f64             ┆ f64             ┆ f64             ┆ f64             ┆ f64             │
╞════════╪═════════════════╪═════════════════╪═════════════════╪═════════════════╪═════════════════╡
│ 0      ┆ 0.474453        ┆ 0.071749        ┆ 0.486486        ┆ 0.181835        ┆ 0.121159        │
│ 0      ┆ 0.321168        ┆ 0.067265        ┆ 0.306306        ┆ 0.077645        ┆ 0.098167        │
│ 0      ┆ 0.583942        ┆ 0.107623        ┆ 0.585586        ┆ 0.240701        ┆ 0.079634        │
│ 0      ┆ 0.452555        ┆ 0.098655        ┆ 0.486486        ┆ 0.182557    

In [18]:
cols = ["Length_normalized", "Height_normalized", "Diameter_normalized", "Whole weight_normalized", "Density_normalized"]

df = lf_normalized.select(cols).collect()

corr = pl.DataFrame(
    {
        c1: [
            df.select(pl.corr(c1, c2)).item()
            for c2 in cols
        ]
        for c1 in cols
    }
)

print(corr)

shape: (5, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ Length_normalized ┆ Height_normalized ┆ Diameter_normaliz ┆ Whole             ┆ Density_normaliz │
│ ---               ┆ ---               ┆ ed                ┆ weight_normalized ┆ ed               │
│ f64               ┆ f64               ┆ ---               ┆ ---               ┆ ---              │
│                   ┆                   ┆ f64               ┆ f64               ┆ f64              │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪══════════════════╡
│ 1.0               ┆ 0.821897          ┆ 0.98661           ┆ 0.928724          ┆ -0.171977        │
│ 0.821897          ┆ 1.0               ┆ 0.827503          ┆ 0.815313          ┆ -0.312288        │
│ 0.98661           ┆ 0.827503          ┆ 1.0               ┆ 0.927886          ┆ -0.182711        │
│ 0.928724          ┆ 0.815313          ┆ 0.927886          ┆ 1.0            